<a href="https://colab.research.google.com/github/namitasathish/MoodPlay/blob/main/MoodPlay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

class SpotifyMoodMusic:
    def __init__(self):
        self.client_id = os.getenv('SPOTIFY_CLIENT_ID')
        self.client_secret = os.getenv('SPOTIFY_CLIENT_SECRET')

        if not self.client_id or not self.client_secret:
            raise ValueError("Missing SPOTIFY_CLIENT_ID or SPOTIFY_CLIENT_SECRET in .env file.")

        self.token = self._get_spotify_token()

    def _get_spotify_token(self):
        try:
            auth_response = requests.post(
                'https://accounts.spotify.com/api/token',
                data={
                    'grant_type': 'client_credentials',
                    'client_id': self.client_id,
                    'client_secret': self.client_secret,
                }
            )
            auth_response.raise_for_status()
            return auth_response.json().get('access_token')
        except requests.exceptions.RequestException as e:
            raise RuntimeError(f"Failed to get Spotify token: {e}")

    def fetch_songs(self, genre, limit=10):
        headers = {'Authorization': f'Bearer {self.token}'}
        url = f'https://api.spotify.com/v1/recommendations?seed_genres={genre}&limit={limit}'

        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            tracks = response.json().get('tracks', [])
            if not tracks:
                print("No tracks found for the given genre.")
                return []
            return [{'title': track['name'], 'artist': track['artists'][0]['name']} for track in tracks]
        except requests.exceptions.RequestException as e:
            print(f"Error fetching songs: {e}")
            return []

    @staticmethod
    def mood_to_genre(mood):
        genremapping = {
            'happy': 'pop',
            'sad': 'sad',
            'energetic': 'dance',
            'relaxed': 'ambient',
            'angry': 'metal',
            'romantic': 'romance',
            'intrigued': 'indie',
            'b': 'soul',
            'urgent': 'electronic',
            'confident': 'rock',
            'reflective': 'soul',
            'frustrated': 'punk',
            'melancholic': 'indie',
            'thoughtful': 'jazz',
            'motivated': 'hip-hop',
            'sentimental': 'acoustic',
            'dreamy': 'indie',
            'playful': 'funk'
        }
        return genremapping.get(mood.lower())

def main():
    print("MoodPlay\n")

    moodmusic = SpotifyMoodMusic()

    while True:
        usermood = input("Enter your mood (e.g., happy, sad, relaxed, confident): ").strip().lower()
        genre = moodmusic.mood_to_genre(usermood)

        if not genre:
            print("Mood not recognized. Try again from supported moods.")
            continue

        try:
            limit = int(input("How many songs would you like to get? (default is 10): ") or 10)
        except ValueError:
            limit = 10

        songs = moodmusic.fetch_songs(genre, limit)

        if songs:
            print(f"\nRecommended songs for your mood '{usermood.title()}':\n")
            for idx, song in enumerate(songs, 1):
                print(f"{idx}. {song['title']} by {song['artist']}")
        else:
            print("No songs could be fetched at this time.")

        retry = input("\nWould you like to try another mood? (y/n): ").strip().lower()
        if retry != 'y':
            print("\nThank you for using MoodPlay!")
            break

if __name__ == '__main__':
    main()


Enter your mood (happy, sad, energetic, relaxed, etc.): confident
Recommended songs for your mood 'confident':

All Along the Watchtower by Jimi Hendrix
By the Way by Red Hot Chili Peppers
Any Way You Want It by Journey
Should I Stay or Should I Go - Remastered by The Clash
Turn by The Wombats
What's Wrong by PVRIS
Paint It Black by The Rolling Stones
Black Magic Woman - Single Version by Santana
Addicted by Saving Abel
Sharp Dressed Man - 2008 Remaster by ZZ Top
